In [17]:
!pip install timm torch torchvision scikit-learn matplotlib seaborn

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [18]:
!pip install h5py

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [19]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import timm
import warnings
warnings.filterwarnings('ignore')

In [20]:
# ============================================================================
# STEP 3: Set Random Seeds for Reproducibility
# ============================================================================
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [21]:
# ============================================================================
# STEP 4: Define Dataset Paths
# ============================================================================
# Update these paths to the actual location of your dataset
train_dir = 'NiAD/NiAD-large/NiAD-large/Train'
val_dir = 'NiAD/NiAD-large/NiAD-large/Val'
test_dir = 'NiAD/NiAD-large/NiAD-large/Test'

# Example of how to specify a path if your dataset is in a different structure
# train_dir = '/content/drive/MyDrive/MyDataset/train'
# val_dir = '/content/drive/MyDrive/MyDataset/val'
# test_dir = '/content/drive/MyDrive/MyDataset/test'

# Assuming 'NiAD-large' is a directory containing 'Train', 'Val', and 'Test' subdirectories
# and each of those contains 'normal' and 'anomalous' subdirectories.
# If your structure is different, please update the paths accordingly.

In [22]:
# ============================================================================
# STEP 5: Channel Attention Module (Squeeze-and-Excitation)
# ============================================================================
class ChannelAttention(nn.Module):
    """
    Channel Attention Module (SE-Block) - focuses on 'what' is meaningful
    Applies attention to feature channels
    """
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (batch_size, channels)
        # Compute channel attention weights
        attention = self.fc(x)
        # Apply attention
        return x * attention


# NOTE: This is the conceptual SE-Block definition shown here for reference.
# The operative class (used by the model) is redefined with the full model
# in the cell below (STEP 7 — SwinWithChannelAttention).


In [23]:
# ============================================================================
# STEP 6: Custom Dataset Class
# ============================================================================
class AnomalyDataset(Dataset):
    """
    Custom Dataset for loading normal and anomalous images
    Expected structure:
        root_dir/
            ├── normal/
            └── anomalous/
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_names = ['Normal', 'Anomalous']

        # Load normal images (label 0)
        normal_dir = os.path.join(root_dir, 'Normal')
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(normal_dir, img_name))
                    self.labels.append(0)

        # Load anomalous images (label 1)
        anomalous_dir = os.path.join(root_dir, 'Anomalous')
        if os.path.exists(anomalous_dir):
            for img_name in os.listdir(anomalous_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                    self.images.append(os.path.join(anomalous_dir, img_name))
                    self.labels.append(1)

        print(f"Loaded {len(self.images)} images from {root_dir}")
        print(f"Normal: {self.labels.count(0)}, Anomalous: {self.labels.count(1)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image if loading fails
            image = Image.new('RGB', (224, 224))

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [24]:
import torch
import torch.nn as nn
import timm


# ============================================================================
# Channel Attention Module (SE-Block)
# Architecture: Global Avg Pool → MLP (Linear-ReLU-Linear-Sigmoid) → scale
# ============================================================================
class ChannelAttention(nn.Module):
    """SE-Block: squeeze to 1×1×C via GAP, then channel-wise MLP + sigmoid."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(1, channels // reduction)
        self.att = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, C) — already squeezed by global average pool
        return x * self.att(x)


# ============================================================================
# Main Model: Swin Transformer + Channel Attention (SE-Block) + MLP Classifier
#
# Pipeline matches architecture diagram:
#   Input frames
#   → Swin Transformer (hierarchical patch merging + transformer blocks)
#   → Global Average Pool  [squeeze W'×H'×C → 1×1×C]
#   → MLP                  [channel excitation weights]
#   → Channel Attention    [1×1×C weights applied to features]
#   → Features with Channel Attention
#   → FC Classifier        [→ Predictions]
# ============================================================================
class SwinWithChannelAttention(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, reduction_ratio=16):
        super().__init__()

        # ── Swin Transformer backbone ──────────────────────────────────────
        # num_classes=0 → raw feature output, no classification head
        self.swin = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0
        )

        # ── Global Average Pool (squeeze 1×1×C) ───────────────────────────
        # Matches diagram: "Global Average Pooling / Squeeze 1×1×C"
        # Named self.global_pool so SwinFeatureExtractor can reference it
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Swin-Base output channels = 1024
        feature_channels = 1024

        # ── Channel Attention SE-Block ─────────────────────────────────────
        # Named self.channel_attention (not self.ca) so:
        #   (a) the unfreeze condition "channel_attention" in name works, and
        #   (b) SwinFeatureExtractor can copy trained_model.channel_attention
        self.channel_attention = ChannelAttention(feature_channels, reduction=reduction_ratio)

        # ── MLP Classifier ─────────────────────────────────────────────────
        # Built here in __init__ (not lazily inside forward) so the optimizer
        # receives all parameters before the first training step.
        self.classifier = nn.Sequential(
            nn.Linear(feature_channels, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat = self.swin(x)   # → (B, seq_len, C) for Swin

        # Normalise output shape to (B, C)
        if feat.dim() == 3:
            # (B, seq_len, C) from Swin: mean-pool over spatial sequence
            feat = feat.mean(dim=1)
        elif feat.dim() == 4:
            # (B, C, H, W): global average pool then flatten
            feat = self.global_pool(feat).flatten(1)

        # SE-Block: channel attention
        feat = self.channel_attention(feat)   # (B, C)

        # Classification head
        return self.classifier(feat)          # (B, num_classes)


In [25]:
# STEP 8: Define Data Transforms
# ============================================================================
print("\nDefining data transforms...")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    # VerticalFlip removed — meaningless for surveillance/anomaly imagery
    transforms.RandomRotation(10),                        # was 15°
    transforms.ColorJitter(brightness=0.1, contrast=0.1,  # was 0.2/0.2/0.2/0.1
                           saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # was 0.1
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


Defining data transforms...


In [26]:
# ============================================================================
# STEP 9: Load Datasets
# ============================================================================
print("\nLoading datasets...")

train_dataset = AnomalyDataset(train_dir, transform=train_transform)
val_dataset = AnomalyDataset(val_dir, transform=val_test_transform)
test_dataset = AnomalyDataset(test_dir, transform=val_test_transform)



Loading datasets...
Loaded 11519 images from NiAD/NiAD-large/NiAD-large/Train
Normal: 7558, Anomalous: 3961
Loaded 877 images from NiAD/NiAD-large/NiAD-large/Val
Normal: 577, Anomalous: 300
Loaded 858 images from NiAD/NiAD-large/NiAD-large/Test
Normal: 540, Anomalous: 318


In [27]:
# ============================================================================
# STEP 10: Create DataLoaders
# ============================================================================
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score

batch_size = 32
num_workers = 2

# ── WeightedRandomSampler: compensates for class imbalance ──────────────
labels_array = np.array(train_dataset.labels)
class_counts  = np.bincount(labels_array)           # [n_normal, n_anomalous]
class_weights = 1.0 / class_counts.astype(float)   # inverse-frequency weights
sample_weights = class_weights[labels_array]        # per-sample weight vector

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,          # replaces shuffle=True
    num_workers=num_workers,
    pin_memory=True
)
print(f"WeightedRandomSampler: class counts {class_counts}, weights {class_weights.round(4)}")

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"\nDataset Statistics:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Dataset Statistics:
Train batches: 360
Val batches: 28
Test batches: 27


In [28]:
# ============================================================================
# Focal Loss — drop-in replacement for CrossEntropyLoss
# ============================================================================
# FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
#
# Key hyperparameters:
#   alpha  : scalar or list[float] of per-class weights (None = uniform)
#            For imbalanced data set alpha=[weight_normal, weight_anomalous]
#   gamma  : focusing parameter.  0 → standard CE.  Typical: 1.0 – 3.0
#   reduction: 'mean' | 'sum' | 'none'
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss for classification.

    Parameters
    ----------
    alpha  : None | float | list[float]
             Per-class weight.  None = 1.0 for all classes.
             Example for binary anomaly detection: [0.25, 0.75]
    gamma  : float  (default 2.0)
             Focusing exponent.  Higher = more focus on hard examples.
    reduction : 'mean' | 'sum' | 'none'
    """
    def __init__(self, alpha=None, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction

        if alpha is None:
            self.alpha = None
        elif isinstance(alpha, (float, int)):
            self.alpha = torch.tensor([alpha, 1.0 - alpha])  # binary shorthand
        else:
            self.alpha = torch.tensor(alpha, dtype=torch.float32)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        logits  : (B, C)  raw model outputs (before softmax)
        targets : (B,)    integer class indices
        """
        # Standard cross-entropy (element-wise, no reduction)
        log_probs = F.log_softmax(logits, dim=-1)              # (B, C)
        ce_loss   = F.nll_loss(log_probs, targets, reduction='none')  # (B,)

        # Probability of the correct class
        probs   = torch.exp(log_probs)                         # (B, C)
        p_t     = probs.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)  # (B,)

        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1.0 - p_t) ** self.gamma

        # Optional per-class alpha weighting
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]    # (B,)
            focal_weight = alpha_t * focal_weight

        loss = focal_weight * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


print("✓ FocalLoss class defined.")


✓ FocalLoss class defined.


In [29]:
# ============================================================================
# STEP 11: Initialize Model, Loss, and Optimizer
# ============================================================================
print("\nInitializing model...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# All layers are created in __init__ — no lazy forward-pass build
model = SwinWithChannelAttention(num_classes=2, pretrained=True, reduction_ratio=16)
model = model.to(device)

# ----------------------------------------------------------------------------
# Selective freezing: keep early Swin stages frozen, unfreeze last 2 stages
# for domain adaptation + always unfreeze channel_attention + classifier.
#
# Why partial unfreeze?
#   - Fully frozen backbone → features are pure ImageNet, not surveillance-domain
#   - Unfreezing layers.2 & layers.3 adapts high-level spatial semantics
#   - Early stages (layers.0, layers.1) stay frozen → preserve low-level filters
# ----------------------------------------------------------------------------

# Step 1: freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: unfreeze Swin last 2 stages
for name, param in model.swin.named_parameters():
    if name.startswith("layers.2") or name.startswith("layers.3"):
        param.requires_grad = True

# Step 3: always unfreeze channel_attention + classifier
for name, param in model.named_parameters():
    if "channel_attention" in name or "classifier" in name:
        param.requires_grad = True

# ----------------------------------------------------------------------------
# Report trainable parameter counts
# ----------------------------------------------------------------------------
trainable_params = [name for name, p in model.named_parameters() if p.requires_grad]
total_params     = sum(p.numel() for p in model.parameters())
trainable_count  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_count:,}  ({100*trainable_count/total_params:.1f}%)")
print(f"Trainable layers    : {len(trainable_params)}")

# ----------------------------------------------------------------------------
# Loss — reduced label_smoothing (0.05 instead of 0.1)
# 0.1 hurts a partially-frozen model that is still finding its footing;
# 0.05 keeps the soft-target benefit without over-penalising correct logits.
# ----------------------------------------------------------------------------
criterion = FocalLoss(
    alpha=[0.345, 0.655],  # exact inverse-frequency weights (not guessed)
    gamma=2.0,    # 2.0 is a good default; try 1.0–3.0
    reduction='mean'
)

# ----------------------------------------------------------------------------
# Optimizer — separate learning-rate groups:
#   • Swin fine-tune layers  → lr=1e-5  (gentle nudge, backbone is fragile)
#   • channel_attention + classifier → lr=1e-4  (head trains at normal speed)
#   • weight_decay 1e-4 (was 5e-3 which over-regularised the tiny head)
# ----------------------------------------------------------------------------
swin_finetune_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and ("swin.layers.2" in n or "swin.layers.3" in n)
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and ("channel_attention" in n or "classifier" in n)
]

optimizer = optim.AdamW([
    {"params": swin_finetune_params, "lr": 1e-5, "weight_decay": 1e-4},  # was 1e-4; gentler backbone fine-tune
    {"params": head_params,          "lr": 1e-4, "weight_decay": 1e-4},  # was 1e-3; avoids head divergence
])

# Scheduler — cosine annealing over 50 epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-7)

print(f"\nOptimizer param groups:")
print(f"  Swin fine-tune  : {len(swin_finetune_params)} tensors  lr=1e-5")
print(f"  Head (CA+clf)   : {len(head_params)} tensors  lr=1e-4")



Initializing model...
Using device: cuda


Total parameters    : 87,532,474
Trainable parameters: 85,411,682  (97.6%)
Trainable layers    : 278

Optimizer param groups:
  Swin fine-tune  : 266 tensors  lr=1e-5
  Head (CA+clf)   : 12 tensors  lr=1e-4


In [30]:
# ============================================================================
# STEP 12: Training Function
# ============================================================================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc  # raw accuracy for quick progress display

In [31]:
# ============================================================================
# STEP 13: Validation Function
# ============================================================================
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc  = 100 * correct / total
    bal_acc    = 100 * balanced_accuracy_score(all_labels, all_predictions)
    return epoch_loss, epoch_acc, bal_acc, all_predictions, all_labels

In [ ]:
# ============================================================================
# STEP 14: Training Loop
# ============================================================================
num_epochs = 100
best_val_acc = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("\n" + "="*70)
print("Starting Training...")
print("="*70)

for epoch in range(num_epochs):
    print(f'\nEpoch [{epoch+1}/{num_epochs}]')
    print('-' * 70)

    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validate
    val_loss, val_acc, val_bal_acc, _, _ = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Print metrics
    print(f'\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%  | Val Bal-Acc: {val_bal_acc:.2f}%')
    print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.2e}')

    # Save best model
    if val_bal_acc > best_val_acc:  # use balanced accuracy as early-stopping criterion
        best_val_acc = val_bal_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss
        }, 'best_swin_channel_attention_model.pth')
        print(f'✓ Best model saved! Val Bal-Acc: {val_bal_acc:.2f}%  (raw: {val_acc:.2f}%)')

    scheduler.step()

    # Early stopping check
   # if epoch > 20 and val_acc < best_val_acc - 5:
    #    print(f"\nEarly stopping triggered. Best Val Acc: {best_val_acc:.2f}%")
     #   break

print("\n" + "="*70)
print("Training Completed!")
print("="*70)



Starting Training...

Epoch [1/100]
----------------------------------------------------------------------


Validation: 100%|██████████████████████████████████████████████| 28/28 [00:05<00:00,  5.57it/s, loss=0.3099, acc=50.29%]



Train Loss: 0.0121 | Train Acc: 97.13%
Val Loss: 0.2206 | Val Acc: 50.29%
Learning Rate: 1.00e-04
✓ Best model saved! Val Acc: 50.29%

Epoch [2/100]
----------------------------------------------------------------------


Validation: 100%|██████████████████████████████████████████████| 28/28 [00:05<00:00,  5.50it/s, loss=0.8621, acc=47.66%]



Train Loss: 0.0056 | Train Acc: 98.50%
Val Loss: 0.4648 | Val Acc: 47.66%
Learning Rate: 9.99e-05

Epoch [3/100]
----------------------------------------------------------------------


...

Validation: 100%|██████████████████████████████████████████████| 28/28 [00:04<00:00,  5.64it/s, loss=1.2397, acc=52.91%]



Train Loss: 0.0002 | Train Acc: 99.93%
Val Loss: 1.1081 | Val Acc: 52.91%
Learning Rate: 2.88e-05

Epoch [70/100]
----------------------------------------------------------------------


Validation: 100%|██████████████████████████████████████████████| 28/28 [00:05<00:00,  5.46it/s, loss=0.4047, acc=49.03%]



Train Loss: 0.0006 | Train Acc: 99.81%
Val Loss: 1.0490 | Val Acc: 49.03%
Learning Rate: 3.17e-05

Epoch [71/100]
----------------------------------------------------------------------


Training:  26%|████████████▍                                  | 95/360 [00:29<01:20,  3.31it/s, loss=0.0000, acc=99.93%]

In [ ]:
# ============================================================================
# STEP 15: Plot Training History
# ============================================================================
print("\nPlotting training history...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', marker='o', markersize=3)
axes[0].plot(val_losses, label='Val Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(train_accs, label='Train Acc', marker='o', markersize=3)
axes[1].plot(val_accs, label='Val Acc', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 16: Load Best Model and Evaluate on Test Set
# ============================================================================
print("\n" + "="*70)
print("Testing Best Model...")
print("="*70)

checkpoint = torch.load('best_swin_channel_attention_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

test_loss, test_acc, test_predictions, test_labels = validate(model, test_loader, criterion, device)

print(f'\n{"="*70}')
print(f"TEST RESULTS")
print(f'{"="*70}')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.2f}%')

In [ ]:
# ============================================================================
# STEP 17: Detailed Classification Metrics
# ============================================================================
print(f'\n{"="*70}')
print("CLASSIFICATION REPORT")
print(f'{"="*70}')
class_names = ['Normal', 'Anomalous']
print(classification_report(test_labels, test_predictions, target_names=class_names, digits=4))

# Precision, Recall, F1 for each class
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_predictions, average=None)
print(f'\nPer-Class Metrics:')
for i, name in enumerate(class_names):
    print(f'{name:12s} - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}, Support: {support[i]}')

# Overall metrics
precision_avg, recall_avg, f1_avg, _ = precision_recall_fscore_support(test_labels, test_predictions, average='macro')
print(f'\nMacro Average - Precision: {precision_avg:.4f}, Recall: {recall_avg:.4f}, F1: {f1_avg:.4f}')


In [ ]:
# ============================================================================
# STEP 18: Confusion Matrix
# ============================================================================
print(f'\n{"="*70}')
print("CONFUSION MATRIX")
print(f'{"="*70}')

cm = confusion_matrix(test_labels, test_predictions)
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix - Test Accuracy: {test_acc:.2f}%')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# STEP 19: Save Final Results
# ============================================================================
results = {
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'classification_report': classification_report(test_labels, test_predictions, target_names=class_names, output_dict=True),
    'confusion_matrix': cm.tolist(),
    'train_losses': train_losses,
    'train_accs': train_accs,
    'val_losses': val_losses,
    'val_accs': val_accs
}

import json
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n" + "="*70)
print("All results saved successfully!")
print("Files created:")
print("  - best_swin_channel_attention_model.pth")
print("  - training_history.png")
print("  - confusion_matrix.png")
print("  - training_results.json")
print("="*70)


In [ ]:
# ============================================================================
# STEP 20: Inference Function for New Images
# ============================================================================
def predict_image(image_path, model, transform, device, class_names=['Normal', 'Anomalous']):
    """
    Predict the class of a single image
    """
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    predicted_class = class_names[predicted.item()]
    confidence_score = confidence.item() * 100

    return predicted_class, confidence_score

# Example usage:
# predicted_class, confidence = predict_image('path/to/image.jpg', model, val_test_transform, device)
# print(f"Predicted: {predicted_class} (Confidence: {confidence:.2f}%)")

print("\n✓ Training pipeline complete! You can now use predict_image() for inference.")

In [ ]:
# ============================================================================
# STEP 21: Save Model Weights to HDF5 File
# ============================================================================
import h5py

hdf5_save_path = 'weights.h5'

try:
    temp_state_dict_path = 'temp_state_dict.pth'
    torch.save(model.state_dict(), temp_state_dict_path)

    with h5py.File(hdf5_save_path, 'w') as f:
        state_dict = torch.load(temp_state_dict_path, map_location=device)
        for key, value in state_dict.items():
            # .detach().clone() releases the PyTorch buffer export before
            # handing the array to h5py — fixes BufferError on re-sized arrays
            arr = value.detach().clone().cpu().numpy()
            f.create_dataset(key, data=arr)

    print(f'\nSuccessfully saved model weights to {hdf5_save_path}')
    os.remove(temp_state_dict_path)

except Exception as e:
    print(f'Error saving model weights to HDF5: {e}')


In [ ]:
# ============================================================================
# Feature Extraction from Swin Transformer with Channel Attention
# ============================================================================
import torch
import torch.nn as nn
import numpy as np
import pickle
from tqdm import tqdm

# ============================================================================
# STEP 1: Create Feature Extractor from Trained Model
# ============================================================================
class SwinFeatureExtractor(nn.Module):
    """
    Extract features from the layer before final classification.
    Returns 256-dimensional features (output of the second-to-last classifier layer).

    FIX: References model attributes by their correct names:
         - trained_model.global_pool      (AdaptiveAvgPool2d)
         - trained_model.channel_attention (SE-Block)
         - trained_model.classifier        (MLP head)
    All three are now defined in SwinWithChannelAttention.__init__.
    """
    def __init__(self, trained_model):
        super(SwinFeatureExtractor, self).__init__()

        self.swin              = trained_model.swin
        self.global_pool       = trained_model.global_pool        # AdaptiveAvgPool2d(1)
        self.channel_attention = trained_model.channel_attention  # SE-Block

        # All classifier layers except the final Linear(256 → num_classes)
        self.feature_layers = nn.Sequential(
            *list(trained_model.classifier.children())[:-1]
        )

    def forward(self, x):
        features = self.swin(x)

        if features.dim() == 4:      # (B, C, H, W)
            features = self.global_pool(features).flatten(1)
        elif features.dim() == 3:    # (B, seq_len, C)
            features = features.mean(dim=1)

        features = self.channel_attention(features)  # SE-Block
        features = self.feature_layers(features)     # → 256-d
        return features

# ============================================================================
# STEP 2: Function to Extract Features for Normal and Anomalous Images
# ============================================================================
def extract_features_by_class(dataloader, feature_extractor, device):
    """
    Extract features and separate by ground truth labels
    Returns:
        normal_features: Features from normal images (label 0)
        anomalous_features: Features from anomalous images (label 1)
    """
    feature_extractor.eval()

    normal_features_list = []
    anomalous_features_list = []

    with torch.no_grad():
        for batch_images, batch_labels in tqdm(dataloader, desc="Extracting features"):
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            # Extract features
            features = feature_extractor(batch_images)
            features = features.cpu().numpy()

            # Separate based on ground truth labels
            batch_labels_np = batch_labels.cpu().numpy()

            # Normal images (label == 0)
            normal_mask = batch_labels_np == 0
            if normal_mask.any():
                normal_features_list.append(features[normal_mask])

            # Anomalous images (label == 1)
            anomalous_mask = batch_labels_np == 1
            if anomalous_mask.any():
                anomalous_features_list.append(features[anomalous_mask])

    # Concatenate all features
    normal_features = np.concatenate(normal_features_list, axis=0) if normal_features_list else np.array([])
    anomalous_features = np.concatenate(anomalous_features_list, axis=0) if anomalous_features_list else np.array([])

    return normal_features, anomalous_features

# ============================================================================
# STEP 3: Save Features to Pickle File
# ============================================================================
def save_features_to_pickle(normal_features, anomalous_features, pickle_file):
    """
    Save extracted features to a pickle file
    """
    data = {
        'normal_features': normal_features,
        'anomalous_features': anomalous_features,
        'feature_dim': normal_features.shape[1] if len(normal_features) > 0 else 0
    }

    with open(pickle_file, 'wb') as f:
        pickle.dump(data, f)

    print(f"\n✓ Features saved to: {pickle_file}")
    print(f"  Normal features shape: {normal_features.shape}")
    print(f"  Anomalous features shape: {anomalous_features.shape}")

# ============================================================================
# STEP 4: Load Features from Pickle File
# ============================================================================
def load_features_from_pickle(pickle_file):
    """
    Load features from a pickle file
    """
    with open(pickle_file, 'rb') as f:
        data = pickle.load(f)

    print(f"\n✓ Features loaded from: {pickle_file}")
    print(f"  Normal features shape: {data['normal_features'].shape}")
    print(f"  Anomalous features shape: {data['anomalous_features'].shape}")
    print(f"  Feature dimension: {data['feature_dim']}")

    return data['normal_features'], data['anomalous_features']

# ============================================================================
# STEP 5: Main Feature Extraction Pipeline
# ============================================================================
def extract_and_save_features(model, train_loader, val_loader, test_loader, device, save_dir='./'):
    """
    Complete pipeline to extract and save features from all datasets
    """
    print("\n" + "="*70)
    print("Creating Feature Extractor...")
    print("="*70)

    # Create feature extractor
    feature_extractor = SwinFeatureExtractor(model)
    feature_extractor = feature_extractor.to(device)
    feature_extractor.eval()

    print(f"Feature extractor created. Output dimension: 256")

    # Extract features from training set
    print("\n" + "="*70)
    print("Extracting Training Set Features...")
    print("="*70)
    train_normal, train_anomalous = extract_features_by_class(train_loader, feature_extractor, device)
    save_features_to_pickle(train_normal, train_anomalous, f'{save_dir}/train_features.pkl')

    # Extract features from validation set
    print("\n" + "="*70)
    print("Extracting Validation Set Features...")
    print("="*70)
    val_normal, val_anomalous = extract_features_by_class(val_loader, feature_extractor, device)
    save_features_to_pickle(val_normal, val_anomalous, f'{save_dir}/val_features.pkl')

    # Extract features from test set
    print("\n" + "="*70)
    print("Extracting Test Set Features...")
    print("="*70)
    test_normal, test_anomalous = extract_features_by_class(test_loader, feature_extractor, device)
    save_features_to_pickle(test_normal, test_anomalous, f'{save_dir}/test_features.pkl')

    print("\n" + "="*70)
    print("Feature Extraction Complete!")
    print("="*70)
    print("\nSummary:")
    print(f"  Training:   {len(train_normal)} normal, {len(train_anomalous)} anomalous")
    print(f"  Validation: {len(val_normal)} normal, {len(val_anomalous)} anomalous")
    print(f"  Test:       {len(test_normal)} normal, {len(test_anomalous)} anomalous")

    return feature_extractor

# ============================================================================
# STEP 6: Run Feature Extraction
# ============================================================================
print("\n" + "="*70)
print("FEATURE EXTRACTION PIPELINE")
print("="*70)

# Load the best trained model
print("\nLoading best trained model...")
checkpoint = torch.load('best_swin_channel_attention_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Model loaded from epoch {checkpoint['epoch']+1}")

# Extract and save features
feature_extractor = extract_and_save_features(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    save_dir='./'
)

# ============================================================================
# STEP 7: Example - Load and Use Extracted Features
# ============================================================================
print("\n" + "="*70)
print("EXAMPLE: Loading Saved Features")
print("="*70)

# Load test features
test_normal, test_anomalous = load_features_from_pickle('test_features.pkl')

print("\nYou can now use these features for:")
print("  - Nearest neighbor search")
print("  - Clustering analysis")
print("  - Visualization (t-SNE, UMAP)")
print("  - Distance-based anomaly detection")
print("  - Building secondary classifiers")

# ============================================================================
# STEP 8: Optional - Compute Feature Statistics
# ============================================================================
def compute_feature_statistics(normal_features, anomalous_features):
    """
    Compute statistics of extracted features
    """
    print("\n" + "="*70)
    print("FEATURE STATISTICS")
    print("="*70)

    print("\nNormal Features:")
    print(f"  Mean: {normal_features.mean(axis=0).mean():.4f}")
    print(f"  Std:  {normal_features.std(axis=0).mean():.4f}")
    print(f"  Min:  {normal_features.min():.4f}")
    print(f"  Max:  {normal_features.max():.4f}")

    print("\nAnomalous Features:")
    print(f"  Mean: {anomalous_features.mean(axis=0).mean():.4f}")
    print(f"  Std:  {anomalous_features.std(axis=0).mean():.4f}")
    print(f"  Min:  {anomalous_features.min():.4f}")
    print(f"  Max:  {anomalous_features.max():.4f}")

    # Compute Euclidean distance between centroids
    normal_centroid = normal_features.mean(axis=0)
    anomalous_centroid = anomalous_features.mean(axis=0)
    centroid_distance = np.linalg.norm(normal_centroid - anomalous_centroid)

    print(f"\nCentroid Distance: {centroid_distance:.4f}")
    print("  (Higher distance = better feature separation)")

# Compute statistics
compute_feature_statistics(test_normal, test_anomalous)

print("\n" + "="*70)
print("✓ Feature extraction pipeline completed successfully!")
print("="*70)

In [ ]:
# ============================================================================
# Load and Process Extracted Features from Pickle Files
# ============================================================================
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

# ============================================================================
# STEP 1: Load Features
# ============================================================================
def load_features_single_file(pickle_file):
    with open(pickle_file, 'rb') as f:
        data = pickle.load(f)

    normal_features = data['normal_features']
    anomalous_features = data['anomalous_features']

    print(f"\n✓ Loaded: {pickle_file}")
    print(f"Normal: {normal_features.shape}, Anomalous: {anomalous_features.shape}")

    return normal_features, anomalous_features

# ============================================================================
# STEP 2: Load Dataset
# ============================================================================
print("="*70)
print("LOADING FEATURES")
print("="*70)

train_normal, train_anomalous = load_features_single_file('train_features.pkl')
val_normal, val_anomalous = load_features_single_file('val_features.pkl')
test_normal, test_anomalous = load_features_single_file('test_features.pkl')

# ============================================================================
# STEP 3: Feature Analysis
# ============================================================================
print("\n" + "="*70)
print("FEATURE ANALYSIS")
print("="*70)

def analyze_features(normal, anomalous, name):
    print(f"\n{name} Dataset")
    print("-"*40)

    normal_centroid = normal.mean(axis=0)
    anomalous_centroid = anomalous.mean(axis=0)

    dist = np.linalg.norm(normal_centroid - anomalous_centroid)

    print(f"Samples → Normal: {len(normal)}, Anomalous: {len(anomalous)}")
    print(f"Centroid Distance: {dist:.4f}")

    return dist

analyze_features(train_normal, train_anomalous, "Train")
analyze_features(val_normal, val_anomalous, "Validation")
analyze_features(test_normal, test_anomalous, "Test")

# ============================================================================
# STEP 4: Distribution Plots
# ============================================================================
print("\n" + "="*70)
print("PLOTTING FEATURE DISTRIBUTIONS")
print("="*70)

def plot_feature_distributions(normal, anomalous):
    plt.figure(figsize=(10,6))
    plt.plot(normal.mean(axis=0), label="Normal")
    plt.plot(anomalous.mean(axis=0), label="Anomalous")
    plt.legend()
    plt.title("Mean Feature Comparison")
    plt.grid(True)
    plt.show()

plot_feature_distributions(test_normal, test_anomalous)

# ============================================================================
# STEP 5: t-SNE (FIXED)
# ============================================================================
print("\n" + "="*70)
print("RUNNING t-SNE")
print("="*70)

def visualize_tsne(normal, anomalous, n_samples=1000):
    n_normal = min(n_samples, len(normal))
    n_anom = min(n_samples, len(anomalous))

    rng = np.random.default_rng(42)

    normal_sample = normal[rng.choice(len(normal), n_normal, replace=False)]
    anomalous_sample = anomalous[rng.choice(len(anomalous), n_anom, replace=False)]

    all_feat = np.vstack([normal_sample, anomalous_sample])
    labels = np.array([0]*n_normal + [1]*n_anom)

    print(f"Running t-SNE on {len(all_feat)} samples...")

    tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
    feat_2d = tsne.fit_transform(all_feat)

    plt.figure(figsize=(8,6))
    plt.scatter(feat_2d[labels==0,0], feat_2d[labels==0,1], alpha=0.5, label="Normal")
    plt.scatter(feat_2d[labels==1,0], feat_2d[labels==1,1], alpha=0.5, label="Anomalous")
    plt.legend()
    plt.title("t-SNE Visualization (Train Set)")
    plt.grid(True)
    plt.show()

# ✅ IMPORTANT: CALL FUNCTION
visualize_tsne(train_normal, train_anomalous)

# ============================================================================
# STEP 6: PCA
# ============================================================================
print("\n" + "="*70)
print("RUNNING PCA")
print("="*70)

def visualize_pca(normal, anomalous):
    all_features = np.vstack([normal, anomalous])
    labels = np.array([0]*len(normal) + [1]*len(anomalous))

    pca = PCA(n_components=2)
    feat_2d = pca.fit_transform(all_features)

    plt.figure(figsize=(8,6))
    plt.scatter(feat_2d[labels==0,0], feat_2d[labels==0,1], alpha=0.5, label="Normal")
    plt.scatter(feat_2d[labels==1,0], feat_2d[labels==1,1], alpha=0.5, label="Anomalous")
    plt.legend()
    plt.title("PCA Visualization")
    plt.grid(True)
    plt.show()

visualize_pca(test_normal, test_anomalous)

# ============================================================================
# STEP 7: Save Features
# ============================================================================
print("\n" + "="*70)
print("SAVING FEATURES")
print("="*70)

with open('all_extracted_features.pkl', 'wb') as f:
    pickle.dump({
        'train_normal': train_normal,
        'train_anomalous': train_anomalous
    }, f)

print("✓ Done!")

In [ ]:
import os
import pandas as pd
from pathlib import Path

def create_csv_from_folders(base_dir, output_csv):
    """
    Create CSV file with image paths and labels
    Normal folder -> label 0
    Anomalous folder -> label 1
    """
    data = []

    # Process normal folder (label 0)
    normal_dir = os.path.join(base_dir, 'Normal') # Corrected folder name
    if os.path.exists(normal_dir):
        print(f"Processing normal folder: {normal_dir}")
        normal_images = [f for f in os.listdir(normal_dir)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

        for img_name in normal_images:
            img_path = os.path.join(normal_dir, img_name)
            data.append({
                'imagepath': img_path,
                'Groundtruth': 0  # Normal = 0
            })
        print(f"  Found {len(normal_images)} normal images")
    else:
        print(f"Warning: Normal folder not found at {normal_dir}")

    # Process anomalous folder (label 1)
    anomalous_dir = os.path.join(base_dir, 'Anomalous') # Corrected folder name
    if os.path.exists(anomalous_dir):
        print(f"Processing anomalous folder: {anomalous_dir}")
        anomalous_images = [f for f in os.listdir(anomalous_dir)
                           if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

        for img_name in anomalous_images:
            img_path = os.path.join(anomalous_dir, img_name)
            data.append({
                'imagepath': img_path,
                'Groundtruth': 1  # Anomalous = 1
            })
        print(f"  Found {len(anomalous_images)} anomalous images")
    else:
        print(f"Warning: Anomalous folder not found at {anomalous_dir}")

    # Create DataFrame
    df = pd.DataFrame(data)

    # Save to CSV
    df.to_csv(output_csv, index=False)

    print(f"\n✓ CSV created successfully!")
    print(f"  Total images: {len(df)}")
    # Only try to access 'Groundtruth' if the DataFrame is not empty
    if not df.empty:
        print(f"  Normal (0): {(df['Groundtruth'] == 0).sum()}")
        print(f"  Anomalous (1): {(df['Groundtruth'] == 1).sum()}")
    print(f"  Saved to: {output_csv}")

    # Show first few rows
    if not df.empty:
        print(f"\nFirst 5 rows:")
        print(df.head())
    else:
        print("\nDataFrame is empty, no rows to display.")


    return df

# Specify your base directory containing 'normal' and 'anomalous' folders
base_dir = 'NIAD-LL/Test'
output_csv = 'NIAD-LL/corrected.csv'

# Create the CSV
df = create_csv_from_folders(base_dir, output_csv)

In [ ]:
# ============================================================================
# PROXIMITY-BASED PREDICTION USING MAHALANOBIS DISTANCE
# ============================================================================
# Pipeline:
#   1. Load train features (Normal + Anomalous) from pickle
#   2. Fit a full-covariance Gaussian per class on train features
#   3. Load test features in batch via DataLoader (no per-image loop)
#   4. Compute scipy Mahalanobis distance to each class centroid
#   5. Predict: class with the smaller distance wins
#   6. Evaluate with sklearn (classification_report, confusion_matrix)
#   7. Plot seaborn confusion matrix heatmap + save results CSV
# ============================================================================

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from scipy.spatial.distance import mahalanobis as scipy_mahalanobis
from scipy.linalg import pinvh
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from tqdm import tqdm

# ── Config ───────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['Normal', 'Anomalous']
REG          = 1e-5          # Tikhonov regularisation added to cov diagonal
SAVE_CSV     = '/content/drive/MyDrive/dataset_S/Target_dataset/mahal_predictions.csv'
SAVE_FIG_CM  = 'confusion_matrix_mahal.png'
SAVE_FIG_DIST = 'mahal_score_distribution.png'

# ── Step 1: Load train features ───────────────────────────────────────────────
print("=" * 70)
print("STEP 1: Loading train features from pickle")
print("=" * 70)
train_normal, train_anomalous = load_features_from_pickle('train_features.pkl')
# shape: (N_normal, 256) and (N_anomalous, 256)
print(f"  Train Normal    : {train_normal.shape}")
print(f"  Train Anomalous : {train_anomalous.shape}")

# ── Step 2: Fit full-covariance Gaussian per class ───────────────────────────
print("\n" + "=" * 70)
print("STEP 2: Fitting full-covariance Gaussians on train features")
print("=" * 70)

def fit_gaussian(features, reg=REG):
    """
    Compute mean and regularised inverse covariance for a class.

    Parameters
    ----------
    features : (N, D) numpy array
    reg      : float — added to diagonal before inversion

    Returns
    -------
    mu : (D,)    — class mean
    VI : (D, D)  — inverse of regularised covariance (via pinvh)
    """
    mu  = features.mean(axis=0)                          # (D,)
    cov = np.cov(features, rowvar=False)                 # (D, D)
    cov += reg * np.eye(cov.shape[0])                    # regularise
    VI  = pinvh(cov)                                     # stable symmetric pseudo-inverse
    return mu, VI

mu_normal,    VI_normal    = fit_gaussian(train_normal)
mu_anomalous, VI_anomalous = fit_gaussian(train_anomalous)

print(f"  Normal    — mu.norm={np.linalg.norm(mu_normal):.4f}  "
      f"VI.shape={VI_normal.shape}")
print(f"  Anomalous — mu.norm={np.linalg.norm(mu_anomalous):.4f}  "
      f"VI.shape={VI_anomalous.shape}")

# ── Step 3: Batch-extract test features via DataLoader ───────────────────────
print("\n" + "=" * 70)
print("STEP 3: Extracting test features in batch")
print("=" * 70)

# Use the already-created feature_extractor (SwinFeatureExtractor)
# and test_loader (val_test_transform, no augmentation)
feature_extractor.eval()

all_features   = []
all_true_labels = []

with torch.no_grad():
    for batch_imgs, batch_labels in tqdm(test_loader, desc="Extracting test features"):
        batch_imgs = batch_imgs.to(device)
        feats      = feature_extractor(batch_imgs)          # (B, 256)
        all_features.append(feats.cpu().numpy())
        all_true_labels.extend(batch_labels.numpy().tolist())

test_features  = np.concatenate(all_features, axis=0)      # (N_test, 256)
true_labels    = np.array(all_true_labels)                  # (N_test,)
print(f"  Test features shape : {test_features.shape}")
print(f"  True labels shape   : {true_labels.shape}  "
      f"(Normal={( true_labels==0).sum()}  Anomalous={(true_labels==1).sum()})")

# ── Step 4: Compute Mahalanobis distances ─────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: Computing scipy Mahalanobis distances")
print("=" * 70)

# Vectorised — one scipy call per sample (fast numpy inner loop)
d_normal    = np.array([scipy_mahalanobis(x, mu_normal,    VI_normal)
                         for x in tqdm(test_features, desc="  dist→Normal")])
d_anomalous = np.array([scipy_mahalanobis(x, mu_anomalous, VI_anomalous)
                         for x in tqdm(test_features, desc="  dist→Anomalous")])

print(f"  d_normal    — mean={d_normal.mean():.4f}  std={d_normal.std():.4f}")
print(f"  d_anomalous — mean={d_anomalous.mean():.4f}  std={d_anomalous.std():.4f}")

# ── Step 5: Predict ───────────────────────────────────────────────────────────
# Assign the class whose centroid is CLOSER (smaller Mahalanobis distance)
predicted_labels = (d_anomalous <= d_normal).astype(int)   # 1=Anomalous, 0=Normal

# Anomaly score: normalised distance ratio (useful for ROC-AUC)
anomaly_score = d_anomalous / (d_normal + d_anomalous + 1e-8)

# ── Step 6: Evaluate with sklearn ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5: Evaluation — sklearn metrics")
print("=" * 70)

acc = accuracy_score(true_labels, predicted_labels)
print(f"\nAccuracy : {acc * 100:.4f}%")

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                             target_names=CLASS_NAMES, digits=4))

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels, predicted_labels, average=None)
print("Per-Class Metrics:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:12s} — Precision: {precision[i]:.4f}  "
          f"Recall: {recall[i]:.4f}  F1: {f1[i]:.4f}  "
          f"Support: {support[i]}")

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='macro')
print(f"\nMacro Average — Precision: {p_macro:.4f}  "
      f"Recall: {r_macro:.4f}  F1: {f1_macro:.4f}")

try:
    auc = roc_auc_score(true_labels, anomaly_score)
    print(f"ROC-AUC        : {auc:.4f}")
except Exception as e:
    print(f"ROC-AUC        : could not compute ({e})")

# ── Step 7a: Confusion matrix — seaborn heatmap ───────────────────────────────
print("\n" + "=" * 70)
print("STEP 6: Confusion Matrix")
print("=" * 70)

cm = confusion_matrix(true_labels, predicted_labels)
print(cm)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label',      fontsize=12)
axes[0].set_title(f'Confusion Matrix\n(Mahalanobis)  Acc={acc*100:.2f}%',
                  fontsize=13, fontweight='bold')

# Right: row-normalised (recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1],
            cbar_kws={'label': 'Recall (row-normalised)'})
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label',      fontsize=12)
axes[1].set_title('Normalised Confusion Matrix\n(row = recall per class)',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(SAVE_FIG_CM, dpi=300, bbox_inches='tight')
plt.show()
print(f"  Saved → {SAVE_FIG_CM}")

# ── Step 7b: Anomaly score distribution ───────────────────────────────────────
plt.figure(figsize=(9, 4))
for cls, name, color in [(0, 'Normal', 'steelblue'), (1, 'Anomalous', 'tomato')]:
    mask = true_labels == cls
    plt.hist(anomaly_score[mask], bins=60, alpha=0.65, density=True,
             label=f'{name} (n={mask.sum()})', color=color)
plt.axvline(0.5, color='black', linestyle='--', alpha=0.6, label='Threshold = 0.5')
plt.xlabel('Anomaly Score  d_anomalous / (d_normal + d_anomalous)', fontsize=11)
plt.ylabel('Density', fontsize=11)
plt.title('Mahalanobis Anomaly Score Distribution — Test Set', fontsize=13,
          fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_FIG_DIST, dpi=300, bbox_inches='tight')
plt.show()
print(f"  Saved → {SAVE_FIG_DIST}")

# ── Step 8: Save results to CSV ───────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 7: Saving results to CSV")
print("=" * 70)

results_df = pd.DataFrame({
    'Groundtruth'         : true_labels,
    'Predicted'           : predicted_labels,
    'AnomalyScore'        : anomaly_score,
    'MahalDistNormal'     : d_normal,
    'MahalDistAnomalous'  : d_anomalous,
})
results_df.to_csv(SAVE_CSV, index=False)
print(f"  Saved → {SAVE_CSV}")
print(f"  Rows  : {len(results_df)}")
print(f"\n✓ Mahalanobis proximity pipeline complete!")
print(f"  Accuracy  : {acc*100:.4f}%")
print(f"  Macro-F1  : {f1_macro:.4f}")
